# Stablecoins — TitusCoin and the Two Ledgers

Welcome to **TitusCoin (`TTC`)**, a completely fictional token targeting **one fictional USD**. Fictional Titus Bank is fictional too; its lawyers are, presumably, on an equally fictional lunch break.

This notebook is a small learning model, not financial advice, a real reserve, or a production stablecoin. We will follow what happens when an issuer creates, moves, redeems, and then under-backs tokens.

In [ ]:
from blockchain_lib.pos import Validator
from blockchain_lib.stablecoin import Blockchain, FiatBackedIssuer, TokenLedger

chain = Blockchain(
    "TitusChain",
    [Validator("Titus-Node-1", 250), Validator("Titus-Node-2", 150)],
)
tituscoin = TokenLedger("TTC", chain)
titus_issuer = FiatBackedIssuer("Fictional Titus Bank", tituscoin)

## 1. Two ledgers, one promise

A fiat-backed token has two different kinds of state:

- The **on-chain token ledger** knows `TTC` balances, total supply, and token events.
- The issuer's **off-chain reserve record** says how many fictional dollars Fictional Titus Bank holds.

The chain cannot peek into a bank vault. Keeping these books separate is the important part: an immaculate chain does not prove the reserve exists. Both begin at zero.

In [ ]:
print(f"Token supply: {tituscoin.total_supply:.0f} TTC")
print(f"Off-chain reserve: ${titus_issuer.reserve_usd:,.0f} fictional USD")

## 2. Issuance: deposit first, mint second

Alice deposits 1,000 fictional USD with the issuer. The issuer records that reserve and mints 1,000 TTC to Alice at the target of one token per fictional dollar. The `MINT` event lands on TitusChain; the reserve entry remains an off-chain claim. No suitcase of dollars is squeezed into a block.

In [ ]:
titus_issuer.deposit_and_mint("Alice", 1_000)

print(f"Alice: {tituscoin.balances['Alice']:.0f} TTC")
print(f"Supply: {tituscoin.total_supply:.0f} TTC")
print(f"Reserve: ${titus_issuer.reserve_usd:,.0f} fictional USD")

## 3. Transfer: ownership moves, supply does not

> **Pause and predict:** Alice sends Bob 250 TTC. What should change: their balances, total supply, the fictional USD reserve, or some combination?

A transfer should only reassign existing tokens. The ledger must subtract and add the same amount, preserving both total supply and the sum of balances. The issuer has no reason to move reserve money because nobody redeemed anything.

In [ ]:
supply_before_transfer = tituscoin.total_supply
reserve_before_transfer = titus_issuer.reserve_usd
transfer_succeeded = tituscoin.transfer("Alice", "Bob", 250)

assert transfer_succeeded
assert tituscoin.total_supply == supply_before_transfer
assert titus_issuer.reserve_usd == reserve_before_transfer
assert sum(tituscoin.balances.values()) == tituscoin.total_supply
print(tituscoin.balances)
print(f"Supply still: {tituscoin.total_supply:.0f} TTC")

## 4. Redemption: burn a token, release a dollar

> **Pause and predict:** Bob redeems 100 TTC. After the operation, what are Bob's balance, total supply, and the issuer's reserve?

Redemption reverses part of issuance. Bob's 100 TTC are burned on-chain, reducing supply, while the issuer releases 100 fictional USD off-chain. In this model those values move together, so fully backed tokens remain fully backed.

In [ ]:
redemption_succeeded = titus_issuer.redeem("Bob", 100)

assert redemption_succeeded
print(f"Bob: {tituscoin.balances['Bob']:.0f} TTC")
print(f"Supply: {tituscoin.total_supply:.0f} TTC")
print(f"Reserve: ${titus_issuer.reserve_usd:,.0f} fictional USD")

## 5. Reserve loss: an off-chain problem does not rewrite the token ledger

> **Pause and predict:** Fictional Titus Bank records a 180 fictional USD reserve loss. Do token balances or total supply automatically shrink? What happens to backing per token?

The loss happens outside TitusChain. Recording it reduces the issuer's reserve figure, but it neither burns Alice's or Bob's tokens nor adds a token transaction. The chain can remain cryptographically valid while the economic promise weakens. Awkward, but educationally useful.

In [ ]:
supply_before_loss = tituscoin.total_supply
balances_before_loss = tituscoin.balances.copy()
blocks_before_loss = len(chain.chain)
loss_recorded = titus_issuer.record_reserve_loss(180)

assert loss_recorded
assert tituscoin.total_supply == supply_before_loss
assert tituscoin.balances == balances_before_loss
assert len(chain.chain) == blocks_before_loss
print(f"Reserve after loss: ${titus_issuer.reserve_usd:,.0f} fictional USD")
print(f"Supply after loss: {tituscoin.total_supply:.0f} TTC")

## 6. Reserve ratio is not market price

The **reserve ratio** is `reserve / token supply`: here, 720 / 900 = 0.80. The model's `backing_per_token` therefore reports 0.80 fictional USD of simplified backing per TTC.

That is an accounting ratio, **not a market price**. A real trading price comes from buyers, sellers, liquidity, redemption confidence, information, and sometimes collective eyebrow-raising. This notebook has no market, order book, oracle, or price-discovery mechanism, so it cannot calculate a depeg price.

In [ ]:
print(f"Reserve ratio: {titus_issuer.reserve_ratio:.0%}")
print(
    f"Simplified backing: ${titus_issuer.backing_per_token:.2f} "
    "fictional USD per TTC"
)
print(f"Chain check: {chain.is_valid()}")

## 7. What this pocket-sized model leaves out

Real fiat-backed tokens need much more than a tidy Python dictionary: custody, independent attestations or audits, legal redemption rights, banking access, access control, identity and compliance processes, operational security, fees, token decimals, failure handling, and trustworthy reporting from off-chain systems.

Our model also assumes every accepted reserve entry is true and every successful redemption pays exactly one fictional USD per token. It demonstrates **state relationships**, not proof of solvency, safety, liquidity, or price stability.

In [ ]:
print("Final fictional snapshot")
print(f"  balances: {tituscoin.balances}")
print(f"  supply: {tituscoin.total_supply:.0f} TTC")
print(f"  reserve: ${titus_issuer.reserve_usd:,.0f}")
print(f"  backing: ${titus_issuer.backing_per_token:.2f} per TTC")

## Next: two chains, one nervous exchange

We can now issue and account for a token on one chain. **Notebook 8 asks the next question: how can Alice and Bob exchange tokens across two independent chains so that both sides settle—or both sides can safely unwind—without trusting one party to go first?**